# IDX-Trade — End-to-End Quant Showcase

**Cross-sectional alpha → frozen decision policy → execution-aware paper portfolio → performance, risk, and uncertainty.**

This is the human-facing flagship notebook for the IDX-Trade E2E system. It is intentionally **read-only**:
it does not fit or tune a model, select features, place orders, mutate PaperState, call providers, or unlock protected outcomes.

```text
IDX market data + PIT / universe
              ↓
         feature engine
              ↓
      frozen V4-X1 ranker
              ↓
          Decision V2
              ↓
           Sizing V1
              ↓
       execution / fills
              ↓
        paper portfolio
              ↓
 alpha quality + portfolio economics + risk + uncertainty
```

### Supported modes

- `HISTORICAL_REPLAY` — consumed/authorized historical E2E replay.
- `FORWARD_LIVE` — live prospective paper state, **outcome-blind**.
- `FORWARD_MATURED` — only an already-authorized canonical evaluation export after the protected-access gate has legitimately opened.

The notebook is a presentation/analysis layer over canonical artifacts. It is **not** a new research runner.

## 0 — Configuration & safety boundary

Point `ARTIFACT_ROOT` at a **prepared read-only showcase bundle**, not directly at provider storage or an outcome vault.

Expected bundle layout:

```text
<root>/
  metadata.json
  sessions.csv
  scores.csv
  decisions.csv
  orders.csv
  fills.csv
  positions.csv
  nav.csv
  execution.csv
  benchmark.csv
  integrity.json
  alpha_outcomes.csv      # HISTORICAL_REPLAY / authorized FORWARD_MATURED only
```

Not every optional file must exist. The notebook degrades gracefully and labels unavailable sections.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

MODE = os.getenv("IDX_TRADE_SHOWCASE_MODE", "HISTORICAL_REPLAY").strip().upper()
ARTIFACT_ROOT = Path(os.getenv("IDX_TRADE_SHOWCASE_ROOT", "CHANGE_ME")).expanduser()

VALID_MODES = {"HISTORICAL_REPLAY", "FORWARD_LIVE", "FORWARD_MATURED"}
if MODE not in VALID_MODES:
    raise ValueError(f"MODE must be one of {sorted(VALID_MODES)}, got {MODE!r}")

# Frozen confirmatory uncertainty settings from the canonical evaluator.
BOOTSTRAP_BLOCK_LENGTH = 5
BOOTSTRAP_REPLICATES = 10_000
BOOTSTRAP_SEED = 20_260_824
ANNUALIZATION_SESSIONS = 252

print("mode:", MODE)
print("artifact root:", ARTIFACT_ROOT)

In [ ]:
# Import the canonical pure metric layer when the repo is on PYTHONPATH.
# If the notebook is opened before the package is installed, local fallback
# helpers below still keep the presentation cells usable.
try:
    from idx_trade.prospective_evaluation_v1 import (
        evaluate_alpha_metrics,
        evaluate_portfolio_metrics,
        evaluate_turnover,
        evaluate_pending_orders,
        evaluate_benchmark,
        nav_daily_returns,
    )
    CANONICAL_EVALUATOR_AVAILABLE = True
except Exception as exc:
    CANONICAL_EVALUATOR_AVAILABLE = False
    CANONICAL_IMPORT_ERROR = exc

print("canonical evaluator available:", CANONICAL_EVALUATOR_AVAILABLE)
if not CANONICAL_EVALUATOR_AVAILABLE:
    print("import note:", repr(CANONICAL_IMPORT_ERROR))

In [ ]:
@dataclass
class ShowcaseBundle:
    root: Path
    metadata: dict
    integrity: dict
    sessions: pd.DataFrame
    scores: pd.DataFrame
    decisions: pd.DataFrame
    orders: pd.DataFrame
    fills: pd.DataFrame
    positions: pd.DataFrame
    nav: pd.DataFrame
    execution: pd.DataFrame
    benchmark: pd.DataFrame
    alpha_outcomes: pd.DataFrame

def _read_json(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def _read_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        return pd.DataFrame()
    return pd.read_csv(path)

def _normalize_session_col(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty or "session_date" not in frame.columns:
        return frame
    out = frame.copy()
    out["session_date"] = pd.to_datetime(out["session_date"], errors="raise").dt.normalize()
    return out

def load_showcase_bundle(root: Path, mode: str) -> ShowcaseBundle:
    if not root.is_dir():
        raise FileNotFoundError(
            f"Prepared showcase bundle not found: {root}. "
            "Set IDX_TRADE_SHOWCASE_ROOT or edit ARTIFACT_ROOT."
        )

    # Strong outcome-blind boundary: live mode never reads outcome-bearing files.
    forbidden_live = [
        root / "alpha_outcomes.csv",
        root / "prospective_evaluation_result.json",
        root / "protected_outcomes.csv",
    ]
    if mode == "FORWARD_LIVE":
        present = [p.name for p in forbidden_live if p.exists()]
        if present:
            raise RuntimeError(
                "FORWARD_LIVE bundle contains outcome-bearing files. "
                f"Use a separate outcome-blind export. Found: {present}"
            )

    def csv(name: str) -> pd.DataFrame:
        return _normalize_session_col(_read_csv(root / name))

    alpha = pd.DataFrame() if mode == "FORWARD_LIVE" else csv("alpha_outcomes.csv")

    return ShowcaseBundle(
        root=root,
        metadata=_read_json(root / "metadata.json"),
        integrity=_read_json(root / "integrity.json"),
        sessions=csv("sessions.csv"),
        scores=csv("scores.csv"),
        decisions=csv("decisions.csv"),
        orders=csv("orders.csv"),
        fills=csv("fills.csv"),
        positions=csv("positions.csv"),
        nav=csv("nav.csv"),
        execution=csv("execution.csv"),
        benchmark=csv("benchmark.csv"),
        alpha_outcomes=alpha,
    )

try:
    run = load_showcase_bundle(ARTIFACT_ROOT, MODE)
    BUNDLE_READY = True
    print("bundle loaded")
except FileNotFoundError as exc:
    run = None
    BUNDLE_READY = False
    display(Markdown(f"**Bundle not loaded yet.** `{exc}`"))

## 1 — Executive snapshot

The first screen should answer four questions quickly:

1. Is there measurable ranking skill?
2. Did the signal survive portfolio construction and execution?
3. What did the resulting paper portfolio earn and risk?
4. Is this historical replay, live outcome-blind monitoring, or matured prospective evidence?

In [ ]:
def _pct(x, digits=2):
    if x is None or not np.isfinite(float(x)):
        return "—"
    return f"{100*float(x):.{digits}f}%"

def _num(x, digits=3):
    if x is None or not np.isfinite(float(x)):
        return "—"
    return f"{float(x):.{digits}f}"

def _portfolio_metrics_fallback(nav: pd.DataFrame) -> dict:
    if nav.empty or not {"session_date", "nav"}.issubset(nav.columns) or len(nav) < 2:
        return {}
    x = nav[["session_date", "nav"]].copy().sort_values("session_date")
    x["nav"] = pd.to_numeric(x["nav"], errors="coerce")
    r = x["nav"].pct_change().dropna().to_numpy(float)
    if len(r) == 0 or not np.isfinite(r).all():
        return {}
    total = float(x["nav"].iloc[-1] / x["nav"].iloc[0] - 1)
    std = float(np.std(r, ddof=1)) if len(r) > 1 else math.nan
    sharpe = float(np.mean(r) / std * np.sqrt(252)) if np.isfinite(std) and std > 0 else math.nan
    peak = x["nav"].cummax()
    dd = x["nav"] / peak - 1
    return {
        "net_total_return": total,
        "annualized_volatility": std*np.sqrt(252) if np.isfinite(std) else math.nan,
        "sharpe_0": sharpe,
        "max_drawdown": float(dd.min()),
    }

if BUNDLE_READY:
    portfolio_metrics = (
        evaluate_portfolio_metrics(run.nav)
        if CANONICAL_EVALUATOR_AVAILABLE and not run.nav.empty
        else _portfolio_metrics_fallback(run.nav)
    )

    alpha_metrics = {}
    if MODE != "FORWARD_LIVE" and not run.alpha_outcomes.empty and CANONICAL_EVALUATOR_AVAILABLE:
        alpha_metrics = evaluate_alpha_metrics(run.alpha_outcomes)

    benchmark_metrics = {}
    if CANONICAL_EVALUATOR_AVAILABLE and not run.nav.empty and not run.benchmark.empty:
        try:
            benchmark_metrics = evaluate_benchmark(run.nav, run.benchmark)
        except Exception:
            benchmark_metrics = {}

    snapshot = pd.DataFrame([
        ["Mode", MODE],
        ["Sessions recorded", len(run.sessions) if not run.sessions.empty else "—"],
        ["Mean Spearman IC", _num(alpha_metrics.get("mean_ic"))],
        ["ICIR", _num(alpha_metrics.get("icir"))],
        ["Net return", _pct(portfolio_metrics.get("net_total_return"))],
        ["Excess vs benchmark", _pct(benchmark_metrics.get("net_excess_return_vs_benchmark"))],
        ["Sharpe (rf=0)", _num(portfolio_metrics.get("sharpe_0"))],
        ["Max drawdown", _pct(portfolio_metrics.get("max_drawdown"))],
    ], columns=["Metric", "Value"])
    display(snapshot)
else:
    display(Markdown("Configure a prepared bundle to populate the executive snapshot."))

## 2 — Alpha quality

**Question:** does the frozen cross-sectional model actually rank subsequent outcomes?

Headline metrics:
- mean daily Spearman IC,
- ICIR,
- positive-IC share,
- 95% moving-block-bootstrap CI,
- rank-bucket monotonicity.

In `FORWARD_LIVE`, this section stays outcome-locked by design.

In [ ]:
if not BUNDLE_READY:
    display(Markdown("Alpha section waiting for a bundle."))
elif MODE == "FORWARD_LIVE":
    display(Markdown("### 🔒 Outcome locked\nLive forward mode does not read realized alpha outcomes."))
elif run.alpha_outcomes.empty:
    display(Markdown("No authorized `alpha_outcomes.csv` is present in this bundle."))
elif not CANONICAL_EVALUATOR_AVAILABLE:
    display(Markdown("Canonical evaluator import is required for alpha metrics."))
else:
    alpha_metrics = evaluate_alpha_metrics(run.alpha_outcomes)
    ic = alpha_metrics["session_ic"].copy()
    ic["rolling_20"] = ic["ic"].rolling(20, min_periods=5).mean()

    headline = pd.DataFrame({
        "Metric": ["Mean IC", "Median IC", "ICIR", "Positive IC", "95% bootstrap CI"],
        "Value": [
            _num(alpha_metrics["mean_ic"]),
            _num(alpha_metrics["median_ic"]),
            _num(alpha_metrics["icir"]),
            _pct(alpha_metrics["positive_ic_fraction"]),
            f"[{alpha_metrics['bootstrap_ci_95'][0]:.4f}, {alpha_metrics['bootstrap_ci_95'][1]:.4f}]",
        ],
    })
    display(headline)

    plt.figure(figsize=(11, 4.5))
    plt.plot(ic["session_date"], ic["ic"], linewidth=1, alpha=.55, label="session IC")
    plt.plot(ic["session_date"], ic["rolling_20"], linewidth=2, label="20-session mean")
    plt.axhline(0, linewidth=1)
    plt.title("Cross-sectional Spearman IC through time")
    plt.xlabel("session")
    plt.ylabel("Spearman IC")
    plt.legend()
    plt.show()

    buckets = alpha_metrics["rank_buckets"]
    order = ["RANK_1_10", "RANK_11_20", "RANK_21_50", "RANK_GT50"]
    labels = ["1–10", "11–20", "21–50", ">50"]
    means = [buckets[k]["mean"] for k in order]

    plt.figure(figsize=(8.5, 4.2))
    plt.bar(labels, means)
    plt.axhline(0, linewidth=1)
    plt.title("Realized outcome by frozen model rank bucket")
    plt.xlabel("rank bucket")
    plt.ylabel("mean realized target")
    plt.show()

## 3 — Signal → portfolio

**Question:** what did the system actually trade?

This section connects ranked scores to decisions, target sizing, orders, and fills.
The notebook displays the artifacts; it never recomputes a new trading policy.

In [ ]:
if not BUNDLE_READY:
    display(Markdown("Signal-to-portfolio section waiting for a bundle."))
else:
    # Latest session table: merge only on keys that are present.
    frames = []
    for name, frame in [
        ("score", run.scores),
        ("decision", run.decisions),
        ("order", run.orders),
        ("fill", run.fills),
    ]:
        if frame.empty or "session_date" not in frame.columns or "ticker" not in frame.columns:
            continue
        f = frame.copy()
        latest = f["session_date"].max()
        f = f.loc[f["session_date"].eq(latest)].copy()
        frames.append((name, f, latest))

    if frames:
        latest_session = max(x[2] for x in frames)
        merged = None
        for name, f, _ in frames:
            f = f.loc[f["session_date"].eq(latest_session)].copy()
            if merged is None:
                merged = f
            else:
                overlap = [
                    c for c in f.columns
                    if c in merged.columns and c not in {"session_date", "ticker"}
                ]
                f = f.rename(columns={c: f"{name}_{c}" for c in overlap})
                merged = merged.merge(f, on=["session_date", "ticker"], how="outer")
        display(Markdown(f"**Latest available session:** `{latest_session.date()}`"))
        show_cols = [
            c for c in [
                "ticker", "rank", "alpha_consensus", "score",
                "decision", "action", "target_weight", "target_notional",
                "filled_notional", "fill_price", "status",
            ] if c in merged.columns
        ]
        if not show_cols:
            show_cols = list(merged.columns[:10])
        display(merged[show_cols].head(20))
    else:
        display(Markdown("No score/decision/order/fill session table is available yet."))

    # Aggregate implementation diagnostics.
    diagnostics = []
    if not run.positions.empty:
        if "ticker" in run.positions.columns and "session_date" in run.positions.columns:
            holdings = (
                run.positions.loc[
                    run.positions.get("position_qty", pd.Series(index=run.positions.index, data=1)).fillna(0).ne(0)
                ]
                .groupby("session_date")["ticker"].nunique()
            )
            if len(holdings):
                diagnostics.append(["Average holdings", float(holdings.mean())])

    if CANONICAL_EVALUATOR_AVAILABLE and not run.execution.empty:
        try:
            t = evaluate_turnover(run.execution)
            diagnostics.append(["Mean daily turnover", t["mean_daily_turnover"]])
            diagnostics.append(["Aggregate turnover", t["aggregate_turnover"]])
        except Exception:
            pass

    if CANONICAL_EVALUATOR_AVAILABLE and not run.orders.empty:
        try:
            p = evaluate_pending_orders(run.orders)
            diagnostics.append(["Pending Open order rate", p["pending_order_rate"]])
        except Exception:
            pass

    if diagnostics:
        display(pd.DataFrame(diagnostics, columns=["Implementation metric", "Value"]))

In [ ]:
# Portfolio breadth / exposure through time.
if BUNDLE_READY and not run.positions.empty and "session_date" in run.positions.columns:
    p = run.positions.copy()

    series_to_plot = {}
    if {"ticker", "position_qty"}.issubset(p.columns):
        active = p.loc[p["position_qty"].fillna(0).ne(0)]
        series_to_plot["holdings"] = active.groupby("session_date")["ticker"].nunique()

    if "market_value" in p.columns:
        series_to_plot["gross market value"] = p.groupby("session_date")["market_value"].apply(
            lambda x: np.abs(pd.to_numeric(x, errors="coerce")).sum()
        )

    if series_to_plot:
        for label, s in series_to_plot.items():
            plt.figure(figsize=(10.5, 3.8))
            plt.plot(s.index, s.values)
            plt.title(f"Portfolio {label} through time")
            plt.xlabel("session")
            plt.ylabel(label)
            plt.show()

## 4 — Paper portfolio performance

**Question:** what economics did the canonical paper portfolio produce?

Primary view:
- paper NAV vs benchmark,
- drawdown,
- cumulative net return,
- Sharpe,
- maximum drawdown.

Annualized metrics are descriptive when the observed horizon is short.

In [ ]:
if not BUNDLE_READY or run.nav.empty:
    display(Markdown("No canonical NAV series is available in this bundle."))
else:
    nav = run.nav[["session_date", "nav"]].copy().sort_values("session_date")
    nav["nav"] = pd.to_numeric(nav["nav"], errors="raise")
    nav["strategy_normalized"] = nav["nav"] / nav["nav"].iloc[0]

    perf = (
        evaluate_portfolio_metrics(nav)
        if CANONICAL_EVALUATOR_AVAILABLE
        else _portfolio_metrics_fallback(nav)
    )

    metric_rows = [
        ["Net total return", _pct(perf.get("net_total_return"))],
        ["Annualized volatility", _pct(perf.get("annualized_volatility"))],
        ["Sharpe (rf=0)", _num(perf.get("sharpe_0"))],
        ["Sortino (rf=0)", _num(perf.get("sortino_0"))],
        ["Max drawdown", _pct(perf.get("max_drawdown"))],
        ["Calmar", _num(perf.get("calmar"))],
    ]
    display(pd.DataFrame(metric_rows, columns=["Portfolio metric", "Value"]))

    plt.figure(figsize=(11, 5))
    plt.plot(nav["session_date"], nav["strategy_normalized"], linewidth=2, label="IDX-Trade paper")

    if not run.benchmark.empty and {"session_date", "benchmark_close"}.issubset(run.benchmark.columns):
        b = run.benchmark[["session_date", "benchmark_close"]].copy().sort_values("session_date")
        b["benchmark_close"] = pd.to_numeric(b["benchmark_close"], errors="coerce")
        b = b.dropna()
        if not b.empty:
            b["benchmark_normalized"] = b["benchmark_close"] / b["benchmark_close"].iloc[0]
            plt.plot(b["session_date"], b["benchmark_normalized"], linewidth=1.7, label="benchmark")

    plt.axhline(1.0, linewidth=1)
    plt.title("Normalized paper NAV vs benchmark")
    plt.xlabel("session")
    plt.ylabel("normalized value")
    plt.legend()
    plt.show()

    nav["running_peak"] = nav["nav"].cummax()
    nav["drawdown"] = nav["nav"] / nav["running_peak"] - 1

    plt.figure(figsize=(11, 3.8))
    plt.plot(nav["session_date"], nav["drawdown"], linewidth=1.6)
    plt.axhline(0, linewidth=1)
    plt.title("Paper portfolio drawdown")
    plt.xlabel("session")
    plt.ylabel("drawdown")
    plt.show()

## 5 — Execution & cost drag

**Question:** did implementation materially consume the economic opportunity?

This section should use canonical execution summaries where available. It does not estimate a replacement slippage model.

In [ ]:
if not BUNDLE_READY or run.execution.empty:
    display(Markdown("No execution summary is available in this bundle."))
else:
    e = run.execution.copy()

    # Standardized names are preferred, but the cell only displays fields that exist.
    candidate_sums = {
        "gross_buy_notional": "Gross buy notional",
        "gross_sell_notional": "Gross sell notional",
        "fees": "Fees",
        "slippage_cost": "Slippage cost",
        "stamp_duty": "Stamp duty",
        "total_cost": "Total trading cost",
    }
    rows = []
    for col, label in candidate_sums.items():
        if col in e.columns:
            v = pd.to_numeric(e[col], errors="coerce")
            if v.notna().any():
                rows.append([label, float(v.sum())])

    if rows:
        display(pd.DataFrame(rows, columns=["Execution quantity", "Total"]))
    else:
        display(Markdown("Execution artifact is present, but no standardized cost columns were found."))

    # Compact gross-to-net economics view when both quantities exist.
    if {"gross_economic_pnl", "net_economic_pnl"}.issubset(e.columns):
        gross = pd.to_numeric(e["gross_economic_pnl"], errors="coerce").sum()
        net = pd.to_numeric(e["net_economic_pnl"], errors="coerce").sum()
        drag = gross - net
        values = [gross, -drag, net]
        labels = ["Gross economics", "Execution drag", "Net economics"]
        plt.figure(figsize=(8.5, 4))
        plt.bar(labels, values)
        plt.axhline(0, linewidth=1)
        plt.title("Gross-to-net execution economics")
        plt.ylabel("P&L / economic units")
        plt.show()

## 6 — Robustness & statistical uncertainty

Instead of a GBM stock-price forecast, the showcase resamples **realized ordered strategy sessions** with a moving-block bootstrap.

Frozen confirmatory settings:
- block length = 5 sessions,
- replicates = 10,000,
- deterministic seed = 20260824.

The simulated horizon equals the observed strategy-return horizon; it is not automatically expanded to 252 sessions.

In [ ]:
def moving_block_paths(
    returns: np.ndarray,
    *,
    replicates: int = BOOTSTRAP_REPLICATES,
    block_length: int = BOOTSTRAP_BLOCK_LENGTH,
    seed: int = BOOTSTRAP_SEED,
) -> np.ndarray:
    r = np.asarray(returns, float)
    if len(r) < block_length:
        raise ValueError(f"Need at least {block_length} ordered returns.")
    rng = np.random.default_rng(seed)
    n = len(r)
    out = np.empty((replicates, n), float)
    block_count = math.ceil(n / block_length)
    for i in range(replicates):
        starts = rng.integers(0, n - block_length + 1, size=block_count)
        idx = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
        out[i] = r[idx]
    return out

if not BUNDLE_READY or run.nav.empty or len(run.nav) < BOOTSTRAP_BLOCK_LENGTH + 1:
    display(Markdown("Need at least 6 NAV markings to show strategy-path uncertainty."))
else:
    nav = run.nav[["session_date", "nav"]].copy().sort_values("session_date")
    r = pd.to_numeric(nav["nav"], errors="raise").pct_change().dropna().to_numpy(float)

    paths_r = moving_block_paths(r)
    paths_nav = np.cumprod(1 + paths_r, axis=1)
    q = np.quantile(paths_nav, [0.05, 0.25, 0.50, 0.75, 0.95], axis=0)
    terminal = paths_nav[:, -1] - 1

    peak = np.maximum.accumulate(paths_nav, axis=1)
    max_dd = (paths_nav / peak - 1).min(axis=1)

    summary = pd.DataFrame({
        "Bootstrap quantity": [
            "P05 terminal return",
            "Median terminal return",
            "P95 terminal return",
            "P(return > 0)",
            "P(max drawdown ≤ -10%)",
            "P(max drawdown ≤ -20%)",
        ],
        "Value": [
            _pct(np.quantile(terminal, .05)),
            _pct(np.median(terminal)),
            _pct(np.quantile(terminal, .95)),
            _pct(np.mean(terminal > 0)),
            _pct(np.mean(max_dd <= -.10)),
            _pct(np.mean(max_dd <= -.20)),
        ],
    })
    display(summary)

    x = np.arange(1, len(r) + 1)
    plt.figure(figsize=(11, 5))
    plt.fill_between(x, q[0], q[4], alpha=.15, label="P05–P95")
    plt.fill_between(x, q[1], q[3], alpha=.25, label="P25–P75")
    plt.plot(x, q[2], linewidth=2, label="median")
    plt.axhline(1.0, linewidth=1)
    plt.title(f"Moving-block bootstrap paper-NAV fan ({len(r)}-session horizon)")
    plt.xlabel("resampled strategy session")
    plt.ylabel("normalized NAV")
    plt.legend()
    plt.show()

    plt.figure(figsize=(9, 4.2))
    plt.hist(terminal, bins=60)
    plt.axvline(0, linewidth=1, label="break-even")
    plt.axvline(np.median(terminal), linewidth=2, label="median")
    plt.title("Bootstrap terminal-return distribution")
    plt.xlabel("terminal return")
    plt.ylabel("replicates")
    plt.legend()
    plt.show()

## 7 — Live forward monitor

This is the part intended to remain useful while the 100-session prospective experiment is still running.

`FORWARD_LIVE` may show:
- recorded/admitted session counts,
- current paper positions and NAV state,
- execution status,
- frozen model identity,
- integrity/CA/input-basis status.

It must **not** show realized forward IC, matured targets, hidden P&L labels, or any protected evaluation result before the access contract permits it.

In [ ]:
if not BUNDLE_READY:
    display(Markdown("Forward monitor waiting for a bundle."))
else:
    progress_rows = []

    if not run.sessions.empty:
        progress_rows.append(["Sessions in bundle", len(run.sessions)])
        if "state" in run.sessions.columns:
            counts = run.sessions["state"].astype(str).value_counts()
            for state, count in counts.items():
                progress_rows.append([f"Session state: {state}", int(count)])

    for key in [
        "model_name",
        "model_generation",
        "model_fingerprint",
        "implementation_commit",
        "target_session_count",
        "status",
    ]:
        if key in run.metadata:
            progress_rows.append([key, run.metadata[key]])

    if progress_rows:
        display(pd.DataFrame(progress_rows, columns=["Forward monitor", "Value"]))
    else:
        display(Markdown("No standardized progress metadata is present."))

    if MODE == "FORWARD_LIVE":
        display(Markdown("### 🔒 Realized alpha outcomes remain locked in this mode."))

## 8 — Research integrity

The notebook should end by making the scientific boundary obvious, not by burying it in a disclaimer.

**Allowed here:** read, hash/check, aggregate, calculate frozen metrics, moving-block bootstrap, visualize.

**Not allowed here:** `model.fit()`, hyperparameter tuning, feature selection, policy optimization, order placement, PaperState mutation, provider calls, counter mutation, protected-outcome discovery/unlock.

In [ ]:
if not BUNDLE_READY:
    display(Markdown("Integrity table will populate from `integrity.json`."))
else:
    preferred = [
        "model_frozen",
        "feature_contract",
        "target_contract",
        "population_contract",
        "outcome_access_policy",
        "execution_provenance",
        "input_basis_integrity",
        "artifact_hashes",
    ]

    rows = []
    for key in preferred:
        if key in run.integrity:
            rows.append([key, run.integrity[key]])

    # Preserve any additional producer-defined integrity fields.
    extras = [k for k in run.integrity.keys() if k not in preferred]
    rows.extend([[k, run.integrity[k]] for k in extras])

    if rows:
        display(pd.DataFrame(rows, columns=["Integrity contract", "Status"]))
    else:
        display(Markdown("`integrity.json` is absent or empty."))

display(Markdown(
    "**Notebook boundary:** this file is a read-only analytical lens over canonical artifacts. "
    "It does not replace the frozen research runners or prospective access gate."
))

---

### Interpretation guide

The flagship question is deliberately simple:

> **Does the alpha rank stocks, does that signal survive portfolio construction and execution, what economics does the resulting paper portfolio produce, and how uncertain are those economics?**

Keep detailed model archaeology, provider debugging, historical candidate comparisons, and forensic audits in supplementary research documents—not in this flagship notebook.